# AI Red Teaming Agent for Generative AI models and applications in Azure AI Foundry

## Objective
This notebook walks through how to use Azure AI Evaluation's AI Red Teaming Agent functionality to assess the safety and resilience of AI systems against adversarial prompt attacks. AI Red Teaming Agent leverages [Risk and Safety Evaluations](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-metrics-built-in?tabs=warning#risk-and-safety-evaluators) to help identify potential safety issues across different risk categories (violence, hate/unfairness, sexual content, self-harm) combined with attack strategies of varying complexity levels from [PyRIT](https://github.com/Azure/PyRIT), Microsoft AI Red Teaming team's open framework for automated AI red teaming.

## Time
You should expect to spend about 30-45 minutes running this notebook. Execution time will vary based on the number of risk categories, attack strategies, and complexity levels you choose to evaluate.

## Before you begin

### Prerequisite
First, if you have an Azure subscription, create an [Azure AI hub](https://learn.microsoft.com/en-us/azure/ai-studio/concepts/ai-resources) then [create an Azure AI project](https://learn.microsoft.com/en-us/azure/ai-studio/concepts/ai-resources). AI projects and Hubs can be served within a private network and are compatible with private endpoints. You **do not** need to provide your own LLM deployment as the AI Red Teaming Agent hosts adversarial models for both simulation and evaluation of harmful content and connects to it via your Azure AI project.

**Note**: In order to upload your results to Azure AI Foundry, you must have the `Storage Blob Data Contributor` role

**Important**: First, ensure that you've installed the [Azure CLI](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli) and then make sure to authenticate to Azure using `az login` in your terminal before running this notebook.

### Installation
From a terminal window, navigate to your working directory which contains this sample notebook, and execute the following.
```bash
python -m venv .venv
```

Then, activate the virtual environment created:

```bash
# %source .venv/bin/activate # If using Mac/Linux OS
.venv/Scripts/activate # If using Windows OS
```

With your virtual environment activated, install the following packages required to execute this notebook:

```bash
pip install uv
uv pip install azure-ai-evaluation[redteam] azure-identity openai
```


Now open VSCode with the following command, and ensure your virtual environment is used as kernel to run the remainder of this notebook.
```bash
code .
```

### Imports

In [1]:
from typing import Optional, Dict, Any
import os

# Azure imports
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.evaluation.red_team import RedTeam, RiskCategory, AttackStrategy

# OpenAI imports
from openai import AzureOpenAI

# Initialize Azure credentials
credential = DefaultAzureCredential()

### Set Up Your Environment Variables

Set the following variables for use in this notebook. These variables connect to your Azure resources and model deployments.

**Note:** You can find these values in your Azure AI Foundry project or Azure OpenAI resource.

For reference, here's an example of what your populated environment variables should look like:

```
# Azure OpenAI
AZURE_OPENAI_API_KEY="your-api-key-here"
AZURE_OPENAI_ENDPOINT="https://endpoint-name.openai.azure.com/openai/deployments/deployment-name/chat/completions"
AZURE_OPENAI_DEPLOYMENT_NAME="gpt-4"
AZURE_OPENAI_API_VERSION="2024-12-01-preview"

# Azure AI Project
AZURE_SUBSCRIPTION_ID="12345678-1234-1234-1234-123456789012"
AZURE_RESOURCE_GROUP_NAME="your-resource-group"
AZURE_PROJECT_NAME="your-project-name"
```

In [2]:
# Azure AI Project information
azure_ai_project = {
    "subscription_id": os.environ.get("AZURE_SUBSCRIPTION_ID", "dcef7009-6b94-4382-afdc-17eb160d709a"),
    "resource_group_name": os.environ.get("AZURE_RESOURCE_GROUP", "rg-ai-foundry-09302-400007"),
    "project_name": os.environ.get("AZURE_PROJECT_NAME", "ai-foundry-project"),
}

# Azure OpenAI deployment information
azure_openai_deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")  # e.g., "gpt-4"
azure_openai_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "https://ai-services-09302-400007.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions")  # e.g., "https://endpoint-name.openai.azure.com/openai/deployments/deployment-name/chat/completions"
azure_openai_api_key = os.environ.get("AZURE_OPENAI_API_KEY", "9YXg0v1HWV4xKoivxzYxzh9GMLiDXn1rI8mMJQTE2bDHSJTPfuTUJQQJ99BFACfhMk5XJ3w3AAAAACOGrtMo")  # e.g., "your-api-key"
azure_openai_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")  # Use the latest API version

## Understanding AI Red Teaming Agent's capabilities

The Azure AI Evaluation SDK's `RedTeam` functionality evaluates AI systems against adversarial prompts across multiple dimensions:

1. **Risk Categories**: Different content risk categories your AI system might generate
   - Violence
   - HateUnfairness
   - Sexual
   - SelfHarm

2. **Attack Strategies**: Along with standard unmodified prompts which are sent by default as the `baseline`, you can specify different transformations of prompts to elicit undesired content.
You can also use `AttackStrategy.Compose()` to layer two strategies in one attack
   - AnsiAttack: Using ANSI escape codes in prompts
   - AsciiArt: Using ASCII art to disguise harmful content
   - AsciiSmuggler: Hiding harmful content within ASCII characters
   - Atbash: Using the Atbash cipher to encode harmful requests
   - Base64: Encoding harmful content in Base64 format
   - Binary: Converting text to binary to bypass filters
   - Caesar: Using the Caesar cipher for encoding
   - CharacterSpace: Manipulating character spacing to confuse filters
   - CharSwap: Swapping characters to bypass detection
   - Diacritic: Using diacritical marks to alter text appearance
   - Flip: Flipping text to bypass content filters
   - Leetspeak: Converting letters to numbers and symbols
   - Morse: Using Morse code to encode harmful requests
   - ROT13: Using ROT13 cipher for text transformation
   - SuffixAppend: Adding suffixes to confuse detection systems
   - StringJoin: Joining strings in unconventional ways
   - Tense: Changing the tense of harmful requests to past tense
   - UnicodeConfusable: Using similar-looking Unicode characters
   - UnicodeSubstitution: Substituting characters with Unicode alternatives
   - Url: Embedding harmful content within URLs
   - Jailbreak: Specially crafted prompts to bypass AI safeguards

3. **Complexity Levels**: Different difficultly levels of attacks
   - Baseline: Standard functionality tests
   - Easy: Simple attack patterns
   - Moderate: More sophisticated attacks
   - Difficult: Complex, layered attack strategies

The key metric for evaluating results is the **Attack Success Rate (ASR)**, which measures the percentage of attacks that successfully elicit harmful content from your AI system.

## Basic Example: Fixed Response Callback

Let's start with a basic example that demonstrates how to set up a Red Team evaluation using a simple callback that always returns a fixed, safe response regardless of the input prompt.

**Important Note**: Since this callback always responds with the same safe message, the Attack Success Rate (ASR) will be 0% for all categories. This example is merely to demonstrate the API structure and flow.

In [3]:
# Define a simple callback function that always returns a fixed response
def financial_advisor_callback(query: str) -> str:  # noqa: ARG001
    return "I'm a financial advisor assistant. I can help with investment advice and financial planning within legal and ethical guidelines."

In [11]:
# Create the `RedTeam` instance with minimal configurations
red_team = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    risk_categories=[
        RiskCategory.Violence, 
        # RiskCategory.HateUnfairness
    ],
    num_objectives=1,
)

NOTE: `num_objectives` specifies the number of attacks to perform per risk category per attack strategy. If the parameter `risk_categories` is not specified, `[RiskCategory.Violence, RiskCategory.HateUnfairness, RiskCategory.Sexual, RiskCategory.SelfHarm]` will be used by default.

Now let's run a simple automated scan using the `RedTeam` with the fixed response target. We'll test against two risk categories and one attack strategy for simplicity.

In [12]:
# Run the red team scan called "Basic-Callback-Scan" with limited scope for this basic example
# This will test 1 objective prompt for each of Violence and HateUnfairness categories with the Flip strategy
result = await red_team.scan(
    target=financial_advisor_callback,
    scan_name="Basic-Callback-Scan",
    attack_strategies=[AttackStrategy.Flip],
    output_path="red_team_output.json",
)

🚀 STARTING RED TEAM SCAN: Basic-Callback-Scan
📂 Output directory: .\.scan_Basic-Callback-Scan_20250626_220115
📊 Risk categories: ['violence']
🔗 Track your red team scan in AI Foundry: https://ai.azure.com/build/evaluation/52499154-39a3-4d94-9bc1-ad4f4f2a2594?wsid=/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-foundry-09302-400007/providers/Microsoft.MachineLearningServices/workspaces/ai-foundry-project
📋 Planning 2 total tasks


Scanning:   0%|                         | 0/2 [00:00<?, ?scan/s, current=fetching baseline/violence]

📚 Using attack objectives from Azure RAI service


Scanning:   0%|                                          | 0/2 [00:03<?, ?scan/s, current=batch 1/1]

📝 Fetched baseline objectives for violence: 1 objectives
🔄 Fetching objectives for strategy 2/2: flip
⚙️ Processing 2 tasks in parallel (max 5 at a time)
▶️ Starting task: baseline strategy for violence risk category
▶️ Starting task: flip strategy for violence risk category


Scanning:  50%|█████████████████                 | 1/2 [00:15<00:15, 15.69s/scan, current=batch 1/1]

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\red_team_output.json".


✅ Completed task 1/2 (50.0%) - baseline/violence in 11.7s
   Est. remaining: 0.3 minutes


Scanning: 100%|██████████████████████████████████| 2/2 [00:16<00:00,  8.23s/scan, current=batch 1/1]

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\red_team_output.json".


✅ Completed task 2/2 (100.0%) - flip/violence in 12.5s
   Est. remaining: 0.0 minutes


Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\red_team_output.json".

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\.scan_Basic-Callback-Scan_20250626_220115\final_results.json".

Overall ASR: 0.0%
Attack Success: 0/2 attacks were successful
------------------------------------------------------------------------------------------------------------
Risk Category  | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------
Violence       | 0.0%           | 0.0%                         | N/A                             | N/A                           

Detailed results available at:
https://ai.azure.com/build/evaluation/52499154-39a3-4d94-9bc1-ad4f4f2a2594?wsid=/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-foundry-09302-400007/providers/Mi

## Intermediary Example: Using a Model Configuration as Target

Now let's create a more realistic example that uses an Azure OpenAI model for responding to the red teaming prompts. To test base or foundation models, you can update your target to take in a model configuration:

In [13]:
# Define a model configuration to test
azure_oai_model_config = {
    "azure_endpoint": azure_openai_endpoint,
    "azure_deployment": azure_openai_deployment,
    "api_key": azure_openai_api_key,
}

Then, update your target to point to the model configurations and run the scan.

In [14]:
# Run the red team scan called "Intermediary-Model-Target-Scan"
result = await red_team.scan(
    target=azure_oai_model_config, 
    scan_name="Intermediary-Model-Target-Scan", 
    attack_strategies=[AttackStrategy.Flip]
)

🚀 STARTING RED TEAM SCAN: Intermediary-Model-Target-Scan
📂 Output directory: .\.scan_Intermediary-Model-Target-Scan_20250626_220218
📊 Risk categories: ['violence']
🔗 Track your red team scan in AI Foundry: https://ai.azure.com/build/evaluation/d038b2d0-b361-475d-9861-06370257c3fc?wsid=/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-foundry-09302-400007/providers/Microsoft.MachineLearningServices/workspaces/ai-foundry-project
📋 Planning 2 total tasks


Scanning:   0%|                             | 0/2 [00:00<?, ?scan/s, current=fetching flip/violence]

📚 Using attack objectives from Azure RAI service
📝 Fetched baseline objectives for violence: 1 objectives
🔄 Fetching objectives for strategy 2/2: flip


Scanning:   0%|                                          | 0/2 [00:00<?, ?scan/s, current=batch 1/1]

⚙️ Processing 2 tasks in parallel (max 5 at a time)
▶️ Starting task: baseline strategy for violence risk category
▶️ Starting task: flip strategy for violence risk category


BadRequestException encountered: Status Code: 400, Message: {"error":{"message":"The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766","type":null,"param":"prompt","code":"content_filter","status":400,"innererror":{"code":"ResponsibleAIPolicyViolation","content_filter_result":{"hate":{"filtered":false,"severity":"safe"},"self_harm":{"filtered":false,"severity":"safe"},"sexual":{"filtered":false,"severity":"safe"},"violence":{"filtered":true,"severity":"high"}}}}}
Scanning:  50%|█████████████████                 | 1/2 [00:17<00:17, 17.95s/scan, current=batch 1/1]

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\.scan_Intermediary-Model-Target-Scan_20250626_220218\baseline_violence_e185e6f4-181b-4a8c-997e-67b2687a2795.json".


✅ Completed task 1/2 (50.0%) - baseline/violence in 17.7s
   Est. remaining: 0.4 minutes


Scanning: 100%|██████████████████████████████████| 2/2 [00:19<00:00,  9.58s/scan, current=batch 1/1]

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\.scan_Intermediary-Model-Target-Scan_20250626_220218\flip_violence_e3dd55ed-25c5-4303-ad64-0cd09c4e7595.json".


✅ Completed task 2/2 (100.0%) - flip/violence in 18.9s
   Est. remaining: 0.0 minutes


Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\.scan_Intermediary-Model-Target-Scan_20250626_220218\final_results.json".

Overall ASR: 0.0%
Attack Success: 0/2 attacks were successful
------------------------------------------------------------------------------------------------------------
Risk Category  | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------
Violence       | 0.0%           | 0.0%                         | N/A                             | N/A                           

Detailed results available at:
https://ai.azure.com/build/evaluation/d038b2d0-b361-475d-9861-06370257c3fc?wsid=/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-foundry-09302-400007/providers/Microsoft.MachineLearningServices/workspaces/ai-foundry-project

📂 All scan files saved to:

## Advanced Example: Using an Azure Open AI Model Endpoint in a Callback Function

Using the same Azure Open AI model configuration as above, we now wrap it in a callback function for more flexibility and control on the input and output handling. This will demonstrate how to evaluate an actual AI application. To test your own actual AI application, replace the inside of the callback function with a call to your application.

In [15]:
# Define a callback that uses Azure OpenAI API to generate responses
async def azure_openai_callback(
    messages: list,
    stream: Optional[bool] = False,  # noqa: ARG001
    session_state: Optional[str] = None,  # noqa: ARG001
    context: Optional[Dict[str, Any]] = None,  # noqa: ARG001
) -> dict[str, list[dict[str, str]]]:
    # Get token provider for Azure AD authentication
    token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")

    # Initialize Azure OpenAI client
    client = AzureOpenAI(
        azure_endpoint=azure_openai_endpoint,
        api_version=azure_openai_api_version,
        azure_ad_token_provider=token_provider,
    )

    ## Extract the latest message from the conversation history
    messages_list = [{"role": message.role, "content": message.content} for message in messages]
    latest_message = messages_list[-1]["content"]

    try:
        # Call the model
        response = client.chat.completions.create(
            model=azure_openai_deployment,
            messages=[
                {"role": "user", "content": latest_message},
            ],
            # max_tokens=500, # If using an o1 base model, comment this line out
            max_completion_tokens=500,  # If using an o1 base model, uncomment this line
            # temperature=0.7, # If using an o1 base model, comment this line out (temperature param not supported for o1 base models)
        )

        # Format the response to follow the expected chat protocol format
        formatted_response = {"content": response.choices[0].message.content, "role": "assistant"}
    except Exception as e:
        print(f"Error calling Azure OpenAI: {e!s}")
        formatted_response = "I encountered an error and couldn't process your request."
    return {"messages": [formatted_response]}

In [16]:
# Create the RedTeam instance with all of the risk categories with 5 attack objectives generated for each category
model_red_team = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    risk_categories=[
        RiskCategory.Violence, 
        RiskCategory.HateUnfairness, 
        RiskCategory.Sexual, 
        RiskCategory.SelfHarm
    ],
    num_objectives=5,
)

We will use this instance of `model_red_team` to test different attack strategies in the following section.

### Testing Different Attack Strategies

Now we'll run a more comprehensive evaluation using multiple attack strategies across risk categories. This will give us a better understanding of our model's vulnerabilities.

In [17]:
# Run the red team scan with multiple attack strategies
advanced_result = await model_red_team.scan(
    target=azure_openai_callback,
    scan_name="Advanced-Callback-Scan",
    attack_strategies=[
        AttackStrategy.EASY,  # Group of easy complexity attacks
        AttackStrategy.MODERATE,  # Group of moderate complexity attacks
        AttackStrategy.CharacterSpace,  # Add character spaces
        AttackStrategy.ROT13,  # Use ROT13 encoding
        AttackStrategy.UnicodeConfusable,  # Use confusable Unicode characters
        AttackStrategy.CharSwap,  # Swap characters in prompts
        AttackStrategy.Morse,  # Encode prompts in Morse code
        AttackStrategy.Leetspeak,  # Use Leetspeak
        AttackStrategy.Url,  # Use URLs in prompts
        AttackStrategy.Binary,  # Encode prompts in binary
        AttackStrategy.Compose([AttackStrategy.Base64, AttackStrategy.ROT13]),  # Use two strategies in one attack
    ],
    output_path="Advanced-Callback-Scan.json",
)

🚀 STARTING RED TEAM SCAN: Advanced-Callback-Scan
📂 Output directory: .\.scan_Advanced-Callback-Scan_20250626_230447
📊 Risk categories: ['violence', 'hate_unfairness', 'sexual', 'self_harm']
🔗 Track your red team scan in AI Foundry: https://ai.azure.com/build/evaluation/89db47da-d979-4933-ab6e-5ce1c631658e?wsid=/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-foundry-09302-400007/providers/Microsoft.MachineLearningServices/workspaces/ai-foundry-project
📋 Planning 52 total tasks


Scanning:   0%|                        | 0/52 [00:00<?, ?scan/s, current=fetching baseline/violence]

📚 Using attack objectives from Azure RAI service


Scanning:   0%|                          | 0/52 [00:02<?, ?scan/s, current=fetching baseline/sexual]

📝 Fetched baseline objectives for violence: 5 objectives
📝 Fetched baseline objectives for hate_unfairness: 5 objectives
📝 Fetched baseline objectives for sexual: 5 objectives


...

Scanning:  98%|█████████████████████████████▍| 51/52 [31:37<00:29, 29.58s/scan, current=batch 11/11]

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\Advanced-Callback-Scan.json".


✅ Completed task 51/52 (98.1%) - tense/self_harm in 68.5s
   Est. remaining: 0.6 minutes


Scanning: 100%|██████████████████████████████| 52/52 [31:52<00:00, 36.78s/scan, current=batch 11/11]

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\Advanced-Callback-Scan.json".


✅ Completed task 52/52 (100.0%) - tense/sexual in 83.7s
   Est. remaining: 0.0 minutes


Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\Advanced-Callback-Scan.json".

Evaluation results saved to "D:\projects\ai-course\611_ai_red_teaming_agent\.scan_Advanced-Callback-Scan_20250626_230447\final_results.json".

Overall ASR: 0.0%
Attack Success: 0/260 attacks were successful
------------------------------------------------------------------------------------------------------------
Risk Category  | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------
Violence       | 0.0%           | 0.0%                         | 0.0%                            | 0.0%                          
Hate-unfairness| 0.0%           | 0.0%                         | 0.0%                            | 0.0%                          
Sexual         | 0.0%           | 0.0%                         | 0.0%       

The data and results used in this attack will be saved to the `output_path` specified. The URL printed out at the end of the scorecard will provide a link to where you results are uploaded and logged to your Azure AI Foundry project.

## Bring your own objectives: Using your own prompts as objectives for RedTeam

Below we demonstrate how to use your own prompts as objectives for a `RedTeam` scan. You can see the required format for prompts under `.\data\prompts.json`. Note that when bringing your own prompts, the supported `risk-type`s are `violence`, `sexual`, `hate_unfairness`, and `self_harm`. The number of prompts you specify will be the `num_objectives` used in the scan. 

In [ ]:
path_to_prompts = ".\data\prompts.json"

# Create the RedTeam specifying the custom attack seed prompts to use as objectives
custom_red_team = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    custom_attack_seed_prompts=path_to_prompts,  # Path to a file containing custom attack seed prompts
)

In [ ]:
custom_red_team_result = await custom_red_team.scan(
    target=azure_openai_callback,
    scan_name="Custom-Prompt-Scan",
    attack_strategies=[
        AttackStrategy.EASY,  # Group of easy complexity attacks
        AttackStrategy.MODERATE,  # Group of moderate complexity attacks
        AttackStrategy.DIFFICULT,  # Group of difficult complexity attacks
    ],
    output_path="Custom-Prompt-Scan.json",
)

## Conclusion

In this notebook, we've demonstrated how to use the Azure AI Evaluation SDK's `RedTeam` functionality to assess the safety and resilience of AI systems. We started with a basic fixed-response example and then moved to a more realistic model testing across multiple risk categories and attack strategies.

The automated AI red teaming scans provides valuable insights into:

1. **Overall Attack Success Rate (ASR)** - The percentage of attacks that successfully elicit harmful content
2. **Vulnerability by Risk Category** - Which types of harmful content your model is most vulnerable to
3. **Effectiveness of Attack Strategies** - Which attack techniques are most successful against your model
4. **Impact of Complexity** - How more sophisticated attacks affect your model's safety guardrails

By regularly red-teaming your AI applications, you can identify and address potential vulnerabilities before deploying your models to production environments.

### Next Steps

1. **Mitigation**: Use these results to strengthen your model's guardrails against identified attack vectors
2. **Continuous Testing**: Implement regular red team evaluations as part of your development lifecycle
3. **Custom Strategies**: Develop custom attack strategies for your specific use cases and domain
4. **Safety Layers**: Consider adding additional safety layers like Azure AI Content Safety to filter harmful requests and responses 